# **01_final_test.ipynb**

**Programmers:**
* Albonia, Jade Lorenz M.
* Caspe, Mark Vincent G.
* Rivera, Rei Djemf M.
* Velante, Kamilah Kaye M.
* Villegas, Jedidiah S.

---

### **System Context**
This notebook executes the final evaluation for the A-EYE Cataract Maturity Classification Tool. It functions as the testing harness to generate the quantitative metrics and visual evidence required for **Chapter 4: Results and Discussion**.

### **Purpose**
To rigorously evaluate the trained A-EYE models against the MobileViT Baseline on the unseen Test Set using a **5-Fold Ensemble strategy**.

---

### **Technical Architecture (Data Structures & Algorithms)**

**1. Data Structures**
* **Model Ensemble (`List[nn.Module]`):** Instead of loading a single weight file, this notebook loads 5 distinct model states (one for each K-Fold split) into memory to perform robust averaging.
* **Prediction Vectors (`np.array`):** Accumulators that store probability scores from all models before final thresholding is applied.

**2. Algorithms**
* **Ensemble Averaging:** Implements a soft voting mechanism where the probability outputs of all 5 fold-models are averaged before classification. This mathematically reduces variance and improves generalization on the Test Set.
* **THOP Profiling:** Uses the `thop` library to  trace the model's forward pass, counting every Multiply-Accumulate (MAC) operation to derive the **GFLOPs** metric.
* **Heuristic Explainability:** A translation logic (in `predict.py`) that converts raw radial token statistics (Mean/Std Dev) into human-readable "Opacity" and "Density" percentages.

**3. Control Flow**
* **Sequential Test Harness:** A linear execution pipeline that evaluates the **Baseline**, then **4-Ring**, **8-Ring**, and **16-Ring** A-EYE variants in order.
* **Automated Archival:** A final routine that compresses the generated confusion matrices and log files into a timestamped ZIP archive and pushes it to Google Drive to prevent data loss.

---



## Step 1: Setup Environment

In [ ]:
import os

GIT_REPO_URL = 'https://github.com/its-levi0sa/A-EYE-Cataract-Maturity-Classification-Tool.git'
PROJECT_DIR_NAME = 'A-EYE'
PROJECT_DIR = os.path.join('/content', PROJECT_DIR_NAME)

# --- Clone or Pull Latest Code ---
if os.path.exists(PROJECT_DIR):
    print("Repository already exists. Pulling latest changes...")
    %cd {PROJECT_DIR}
    !git pull
else:
    print("Cloning repository...")
    !git clone {GIT_REPO_URL} {PROJECT_DIR}
    %cd {PROJECT_DIR}

print("\n✅ Setup complete!\n\n")
print("Current directory:", PROJECT_DIR)
print("\nContents:")
!ls -F

## Step 2: Configure Environment

In [ ]:
import sys
import os
import warnings

# Suppress harmless warnings
warnings.filterwarnings('ignore')

# --- Environment Configuration ---
if PROJECT_DIR not in sys.path:
  sys.path.append(PROJECT_DIR)
  print(f"Added '{PROJECT_DIR}' to Python path.")
else:
  print(f"'{PROJECT_DIR}' is already in the Python path.")

# Install all required packages
print("\nInstalling dependencies...")
!pip install -q -r requirements.txt

print("\n\n✅ Environment configured and dependencies installed!")

## Step 3: Run Quantitative Evaluation

In [ ]:
# --- Change to the project's root directory ---
%cd /content/A-EYE/

# --- Define the path to the TEST data directory ---
TEST_DATA_PATH = 'data/test'

print("--- evaluating BASELINE model ---")
!python -m scripts.evaluate \
    --model_type 'baseline' \
    --model_dir 'saved_models/baseline' \
    --data_dir {TEST_DATA_PATH}

print("\n--- evaluating A-EYE 4-RING model ---")
!python -m scripts.evaluate \
    --model_type 'aeye' \
    --num_rings 4 \
    --model_dir 'saved_models/aeye_4_ring' \
    --data_dir {TEST_DATA_PATH}

print("\n--- evaluating A-EYE 8-RING model ---")
!python -m scripts.evaluate \
    --model_type 'aeye' \
    --num_rings 8 \
    --model_dir 'saved_models/aeye_8_ring' \
    --data_dir {TEST_DATA_PATH}

print("\n--- evaluating A-EYE 16-RING model ---")
!python -m scripts.evaluate \
    --model_type 'aeye' \
    --num_rings 16 \
    --model_dir 'saved_models/aeye_16_ring' \
    --data_dir {TEST_DATA_PATH}

print("\n✅ Quantitative evaluation complete! Check the 'results' folder.")

## Step 4: Run Efficiency Analysis

Here, the FLOPs and Parameters for each model were calculated to provide evidence for their efficiency.

In [ ]:
print("--- Analyzing BASELINE model ---")
!python -m scripts.calculate_flops --model_type baseline

print("\n--- Analyzing A-EYE 4-RING model ---")
!python -m scripts.calculate_flops --model_type aeye --num_rings 4

print("\n--- Analyzing A-EYE 8-RING model ---")
!python -m scripts.calculate_flops --model_type aeye --num_rings 8

print("\n--- Analyzing A-EYE 16-RING model ---")
!python -m scripts.calculate_flops --model_type aeye --num_rings 16

print("\n✅ Efficiency analysis complete!")

## Step 5: Run Qualitative Analysis

In [ ]:
# # --- BASELINE ---
SAMPLE_IMAGE_PATH = "/content/A-EYE/data/test/mature/mature_065.png"

# # --- Run prediction directly on the image file ---
!python -m scripts.predict \
 --model_type 'baseline' \
 --model_dir saved_models/baseline \
 --image_path {SAMPLE_IMAGE_PATH}




# # --- 4 RING ---
SAMPLE_IMAGE_PATH = "/content/A-EYE/data/test/mature/mature_065.png"
BEST_MODEL_RINGS = 4

# # --- Run prediction directly on the image file ---
!python -m scripts.predict \
 --model_type 'aeye' \
 --num_rings {BEST_MODEL_RINGS} \
 --model_dir saved_models/aeye_{BEST_MODEL_RINGS}_ring \
 --image_path {SAMPLE_IMAGE_PATH}

## Step 6: Visualize and Analyze Final Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
import base64
from IPython.display import display, HTML

def parse_eval_file(filepath):
    """Parses a single evaluation log file to extract key metrics."""
    data = {}
    with open(filepath, 'r') as f:
        for line in f:
            if ' - ' in line:
                content = line.split(' - ', 1)[1]
                if ':' in content:
                    try:
                        key, value = content.split(':', 1)
                        key = key.strip()
                        if key in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
                            parsed_value = float(value.strip().split()[0])
                            data[key] = parsed_value
                    except (ValueError, IndexError):
                        continue
    return data

# Find all result files
results_dir = 'results/'
results_files = sorted(glob.glob(os.path.join(results_dir, 'evaluation_results_*.txt')))
all_results = []

for f in results_files:
    model_name = os.path.basename(f).replace('evaluation_results_', '').replace('.txt', '').replace('_', ' ').title()
    model_name = model_name.replace("Aeye", "A-EYE").replace("Results ", "")

    result_data = parse_eval_file(f)
    result_data['Model'] = model_name
    all_results.append(result_data)

df = pd.DataFrame(all_results)

# Reorder columns and set index for display
column_order = ['Model', 'F1-Score', 'Accuracy', 'Precision', 'Recall']
df = df[column_order]

# Order the rows
desired_order = ['Baseline', 'A-EYE 4 Rings', 'A-EYE 8 Rings', 'A-EYE 16 Rings']
df['Model'] = pd.Categorical(df['Model'], categories=desired_order, ordered=True)
df = df.sort_values('Model')

print("--- Final Performance Summary Table ---")
display(df.set_index('Model'))

# --- Color Palette ---
color_palette = ['#FFD700', '#0E1A40', '#941B0C', '#1A472A']

# --- Plotting Performance Metrics ---
metrics_to_plot = ['F1-Score', 'Accuracy', 'Precision', 'Recall']
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Model Performance Comparison on Test Set', fontsize=20, fontweight='bold')
axes = axes.flatten()

for i, metric in enumerate(metrics_to_plot):
    sns.barplot(ax=axes[i], x='Model', y=metric, data=df, palette=color_palette)
    axes[i].set_title(metric, fontsize=14, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Score', fontsize=12)
    axes[i].tick_params(axis='x', rotation=15)
    axes[i].grid(axis='y', linestyle='--', alpha=0.7)

    # Auto-scaling Y-axis
    min_val = df[metric].min() * 0.98
    max_val = df[metric].max() * 1.02
    axes[i].set_ylim(min(0.85, min_val), max(1.0, max_val))

    for container in axes[i].containers:
        axes[i].bar_label(container, fmt='%.4f')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# --- Displaying Confusion Matrices ---
cm_files_raw = glob.glob('results/confusion_matrix_*.png')
order_keys = ['baseline', 'aeye_4_rings', 'aeye_8_rings', 'aeye_16_rings']
cm_files_ordered = []
for name in order_keys:
    for f in cm_files_raw:
        if name in f:
            cm_files_ordered.append(f)
            break
if cm_files_ordered:
    image_html = ""
    for file_path in cm_files_ordered:
        model_name = os.path.basename(file_path).replace('confusion_matrix_evaluation_results_', '').replace('.png', '').replace('_', ' ').title()
        model_name = model_name.replace("Aeye", "A-EYE").replace("Results ", "")

        with open(file_path, "rb") as image_file:
            encoded_string = base64.b64encode(image_file.read()).decode('utf-8')
        image_html += f"""
        <div style="display:inline-block; text-align:center; margin:10px;">
            <p style="font-weight:bold;">{model_name}</p>
            <img src="data:image/png;base64,{encoded_string}" width="300">
        </div>
        """
    display(HTML(f"<h2>Confusion Matrices on Test Set</h2>{image_html}"))
else:
    print("No confusion matrix images found in the 'results' folder.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import precision_recall_curve
from torch.utils.data import DataLoader
import glob
import os
import seaborn as sns

# --- Import your model classes ---
from src.aeye_model import AEyeModel
from src.baseline_model import mobilevit_s
from src.data_utils import get_transforms, AlbumentationsDataset

# --- Configuration ---
sns.set_theme(style="whitegrid", font_scale=1.2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 16

# --- 1. Setup Test Data ---
test_files = glob.glob('data/test/*/*.[jp][pn]g')
test_labels = [0 if 'immature' in f else 1 for f in test_files]

ds = AlbumentationsDataset(
    test_files,
    test_labels,
    transform=get_transforms(is_train=False)
)
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Found {len(test_files)} test images.")

# --- 2. Ensemble Prediction Function ---
def get_ensemble_probs(model_class, weights_dir, config=None):
    weight_paths = glob.glob(os.path.join(weights_dir, "*.pth"))
    if not weight_paths:
        print(f"⚠️ NO WEIGHTS FOUND IN: {weights_dir}")
        return np.array([]), np.array([])

    print(f"--- Ensembling {len(weight_paths)} models from {weights_dir} ---")

    all_fold_probs = []
    true_labels = []

    for i, weight_path in enumerate(weight_paths):
        print(f"  Loading Fold {i+1}: {os.path.basename(weight_path)}")

        if config:
            model = model_class(config)
        else:
            model = model_class()
            model.fc = torch.nn.Linear(model.fc.in_features, 1)

        model.load_state_dict(torch.load(weight_path, map_location=device))
        model.to(device).eval()

        fold_probs = []
        fold_labels = []
        with torch.no_grad():
            for x, y in loader:
                x = x.to(device)
                out = torch.sigmoid(model(x)).cpu().numpy()
                fold_probs.extend(out)
                fold_labels.extend(y.numpy())

        all_fold_probs.append(fold_probs)
        if i == 0:
            true_labels = fold_labels

    avg_probs = np.mean(all_fold_probs, axis=0)
    print("  Ensemble complete.\n")
    return np.array(true_labels), np.array(avg_probs).flatten()

# --- 3. Run Predictions ---
base_weights_dir = 'saved_models/baseline'
y_true, y_base = get_ensemble_probs(mobilevit_s, base_weights_dir)

aeye_config = {'dims': [32, 64, 128, 160], 'embed_dim': 256, 'num_rings': 4}
aeye_weights_dir = 'saved_models/aeye_4_ring'
_, y_aeye = get_ensemble_probs(AEyeModel, aeye_weights_dir, config=aeye_config)

# --- 4. Calculate Curve Data ---
def get_curve_data(y_true, y_scores):
    precision, recall, thresholds = precision_recall_curve(y_true, y_scores)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-6)
    return precision[:-1], recall[:-1], f1[:-1], thresholds

p_b, r_b, f1_b, t_b = get_curve_data(y_true, y_base)
p_a, r_a, f1_a, t_a = get_curve_data(y_true, y_aeye)

# --- 5. Plotting Function (UPDATED STYLE) ---
def plot_metric_vs_confidence(metric_base, metric_aeye, t_base, t_aeye, metric_name):
    plt.figure(figsize=(8, 6))

    # A-EYE: Navy Blue, Solid, Thinner
    plt.plot(t_aeye, metric_aeye, label='A-EYE (4-Ring)', color='navy',
             linestyle='-', linewidth=1)

    # Baseline: Orange, Solid (Fixed from dashed), Thinner
    plt.plot(t_base, metric_base, label='Baseline', color='darkorange',
             linestyle='-', linewidth=1)

    plt.xlabel('Confidence Threshold', fontweight='bold', fontsize=12)
    plt.ylabel(metric_name, fontweight='bold', fontsize=12)
    # plt.title(f'{metric_name} vs. Confidence Threshold', fontsize=14, pad=10)
    plt.legend(loc='best', frameon=True, fancybox=True, shadow=True, fontsize=10)

    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    sns.despine()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    os.makedirs('results', exist_ok=True)
    filename = f"results/{metric_name.lower()}_ensemble_curve_straight.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"✅ Saved plot to {filename}")
    plt.show()

# --- 6. Generate Plots ---
if len(y_base) > 0 and len(y_aeye) > 0:
    plot_metric_vs_confidence(p_b, p_a, t_b, t_a, 'Precision')
    plot_metric_vs_confidence(r_b, r_a, t_b, t_a, 'Recall')
    plot_metric_vs_confidence(f1_b, f1_a, t_b, t_a, 'F1-Score')
else:
    print("❌ Error: Could not generate plots because model predictions were missing.")

## Step 7: Download Final Results

In [ ]:
!zip -r /content/final_aeye_results.zip {PROJECT_DIR}/results

print("✅ All results have been zipped into '/content/final_aeye_results.zip'.")
print("Download it from the file browser on the left before closing this session.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy the zip file into MyDrive
!cp /content/final_aeye_results.zip /content/drive/MyDrive/